In [1]:
import numpy as np
import pandas as pd

c:\Users\Ittikorn\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Ittikorn\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
df = pd.read_csv(r"C:\Users\Ittikorn\OneDrive\Desktop\Ai\Machinelearning\ssswork\Dataset\Raisin_Dataset.csv")

In [4]:
df.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,Extent,Perimeter,Class
0,87524,442.246011,253.291155,0.819738,90546,0.758651,1184.040,Kecimen
1,75166,406.690687,243.032436,0.801805,78789,0.684130,1121.786,Kecimen
2,90856,442.267048,266.328318,0.798354,93717,0.637613,1208.575,Kecimen
3,45928,286.540559,208.760042,0.684989,47336,0.699599,844.162,Kecimen
4,79408,352.190770,290.827533,0.564011,81463,0.792772,1073.251,Kecimen


In [5]:
df.isnull().sum()

Area               0
MajorAxisLength    0
MinorAxisLength    0
Eccentricity       0
ConvexArea         0
Extent             0
Perimeter          0
Class              0
dtype: int64

In [6]:
class Model(nn.Module) :
    def __init__(self,in_feature=7,h1=8,h2=8,h3=8,out_feature=2) : # __init__ สร้างชิ้นส่วนของโมเดล
        super().__init__()
        self.fc1 = nn.Linear(in_feature,h1)
        self.fc2 = nn.Linear(h1,h2)
        self.fc3 = nn.Linear(h2,h3)
        self.out = nn.Linear(h3,out_feature)
        
    def forward(self,x) : #   ฟังก์ชันที่กำหนดว่า ข้อมูลไหลผ่านโมเดลยังไง
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.out(x)
        return x
    

In [7]:
# Seed คือ “ค่าตั้งต้นของความสุ่ม” ในการคำนวณครับ
torch.manual_seed(41)

In [8]:
model = Model()
print(model)

Model(
  (fc1): Linear(in_features=7, out_features=8, bias=True)
  (fc2): Linear(in_features=8, out_features=8, bias=True)
  (fc3): Linear(in_features=8, out_features=8, bias=True)
  (out): Linear(in_features=8, out_features=2, bias=True)
)


In [9]:
# Split X , y
X = df.drop('Class',axis=1)
y = df['Class']

In [10]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X,y,train_size=0.2,random_state=0)

In [11]:
from sklearn.preprocessing import LabelEncoder # Label Encoder คือเครื่องมือที่ใช้แปลง “ข้อมูลประเภทข้อความ (string)” ให้เป็น “ตัวเลข (integer)” เพื่อให้โมเดลเรียนรู้
le = LabelEncoder()
y_train = le.fit_transform(y_train) # Dataframe กลายเป็น numpy.ndarray
y_test = le.fit_transform(y_test) # Dataframe กลายเป็น numpy.ndarray

In [12]:
# Convert Dataframe to Pytorch (จากตารางเป็นตัวเลขล้วน)
X_train = torch.tensor(X_train.values, dtype=torch.float32)
y_train = torch.tensor(y_train,dtype=torch.long) # Long เป็น integer 

X_test = torch.tensor(X_test.values,dtype=torch.float32)
y_test = torch.tensor(y_test,dtype=torch.long)

### เริ่มจากตรงนี้

In [13]:
#DataLoader คือเครื่องมือที่ใช้ “จัดการการส่งข้อมูลเข้าโมเดลทีละชุด (batch)
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [15]:
for epoch in range(100):
    
    for x, y in train_loader:

        # 1. forward
        output = model(x)

        # 2. loss
        loss = criterion(output, y)

        # 3. reset gradient
        optimizer.zero_grad()

        # 4. backprop
        loss.backward()

        # 5. update weight
        optimizer.step()

    print("epoch:", epoch, "loss:", loss.item())

epoch: 0 loss: 16.492223739624023
epoch: 1 loss: 50.515541076660156
epoch: 2 loss: 15.389910697937012
epoch: 3 loss: 20.728961944580078
epoch: 4 loss: 18.54475975036621
epoch: 5 loss: 4.06317663192749
epoch: 6 loss: 15.041765213012695
epoch: 7 loss: 3.2803795337677
epoch: 8 loss: 1.8004642724990845
epoch: 9 loss: 0.9016585350036621
epoch: 10 loss: 1.0839710235595703
epoch: 11 loss: 0.4177124500274658
epoch: 12 loss: 0.639606773853302
epoch: 13 loss: 1.414271354675293
epoch: 14 loss: 1.1258753538131714
epoch: 15 loss: 1.607529640197754
epoch: 16 loss: 1.2576286792755127
epoch: 17 loss: 1.482362985610962
epoch: 18 loss: 1.08147132396698
epoch: 19 loss: 1.1614614725112915
epoch: 20 loss: 0.9671815037727356
epoch: 21 loss: 0.7888566255569458
epoch: 22 loss: 0.869484543800354
epoch: 23 loss: 1.6558917760849
epoch: 24 loss: 1.7134815454483032
epoch: 25 loss: 0.8766714930534363
epoch: 26 loss: 0.7623201012611389
epoch: 27 loss: 3.1199560165405273
epoch: 28 loss: 1.5152833461761475
epoch: 29 l

In [18]:
with torch.no_grad():
    output = model(X_test)
    _, pred = torch.max(output, 1)

    accuracy = (pred == y_test).float().mean()
    print("Accuracy:", accuracy.item())

Accuracy: 0.7055555582046509
